# ⚡ RESULTS SUMMARY

This notebook collects the final results from both implementations (uniform and smart unrecorded teleportation) for quick reference. The actual code is in the separate uniform and non-uniform notebooks.

In [3]:
import igraph as ig
import numpy as np
from infomap import Infomap

from map_equation import (
    compute_description_length,
    compute_description_length_uniform,
)

## Results

In [20]:
def L_official_on_partition(g, partition, weighted):
    im = Infomap(directed=True, two_level=True, silent=True)
    for e in g.es:
        im.add_link(e.source, e.target, e["weight"] if weighted else 1.0)
    im.run(initial_partition={node_id: mod for node_id, mod in enumerate(partition)})
    return im.codelength


print("="*14)
print("✨ UNDIRECTED")
print("="*14)

g = ig.Graph.Read_GraphML("C:/Users/savin/ITI/unweighted_undirected.graphml")
c = g.community_infomap()
print(f"\nunweighted:  custom = {compute_description_length(g, c.membership):.4f}   igraph = {c.codelength:.4f}")

g = ig.Graph.Read_GraphML("C:/Users/savin/ITI/weighted_undirected.graphml")
c = g.community_infomap(edge_weights="weight")
print(f"weighted:    custom = {compute_description_length(g, c.membership):.4f}   igraph = {c.codelength:.4f}")

print("\n" + "="*43)
print("✨ DIRECTED — uniform recorded (+ Laura's)")
print("="*43)

g = ig.Graph.Read_GraphML("C:/Users/savin/ITI/unweighted_directed.graphml")
c = g.community_infomap()
print(f"\nunweighted:  custom = {compute_description_length_uniform(g, c.membership):.4f}   Laura = 3.754")

g = ig.Graph.Read_GraphML("C:/Users/savin/ITI/weighted_directed.graphml")
c = g.community_infomap(edge_weights="weight")
print(f"weighted:    custom = {compute_description_length_uniform(g, c.membership):.4f}   Laura = 3.485")

print("\n" + "="*43)
print("✨ DIRECTED — smart unrecorded (+ paper's)")
print("="*43)

g = ig.Graph.Read_GraphML("C:/Users/savin/ITI/unweighted_directed.graphml")
c = g.community_infomap()
partition = c.membership
print(f"\nunweighted:  custom = {compute_description_length(g, partition):.4f}   official = {L_official_on_partition(g, partition, weighted=False):.4f}   igraph = {c.codelength:.4f}")

g = ig.Graph.Read_GraphML("C:/Users/savin/ITI/weighted_directed.graphml")
c = g.community_infomap(edge_weights="weight")
partition = c.membership
print(f"weighted:    custom = {compute_description_length(g, partition):.4f}   official = {L_official_on_partition(g, partition, weighted=True):.4f}   igraph = {c.codelength:.4f}")

✨ UNDIRECTED

unweighted:  custom = 3.4016   igraph = 3.4016
weighted:    custom = 3.3367   igraph = 3.3367

✨ DIRECTED — uniform recorded (+ Laura's)

unweighted:  custom = 3.7540   Laura = 3.754
weighted:    custom = 3.4955   Laura = 3.485

✨ DIRECTED — smart unrecorded (+ paper's)

unweighted:  custom = 3.0894   official = 3.1215   igraph = 3.5158
weighted:    custom = 2.7365   official = 2.7130   igraph = 3.3920


## Takeaways

- Undirected (both weighted and unweighted): custom L matches igraph exactly (teleportation doesn't matter).
- Directed uniform: custom L match with igraph's (and Laura's).
- Directed smart unrecorded: custom L matches the official infomap package within ~0.7% on the same partition; small residual gap (maybe) from a remaining mismatch in my PageRank computation (max ~0.5% per node) that I haven't fully tracked down, TODO: read more of the C++ src.

📌 the original big difference with igraph on directed (12-25%) wasnt completely a formula issue; most part was because igraph's optimizer finds a different partition than the standalone official infomap package (even though they use the same library underneath) PLUS smth that idk 😭